In [1]:
import hashlib
import json
from dataclasses import dataclass, field
from functools import partial
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

In [2]:
# --- Data configuration ---
@dataclass(frozen=True)
class DataConfig:
    raw_dir: Path
    processed_dir: Path
    csv_path: Path
    img_size: int
    diseases: tuple[str, ...]
    disease_to_idx: dict[str, int] = field(init=False)

    def __post_init__(self):
        object.__setattr__(self, "disease_to_idx", {d: i for i, d in enumerate(self.diseases)})

    @property
    def num_classes(self) -> int:
        return len(self.diseases)

    @property
    def cache_tag(self) -> str:
        """Hash of (disease list, img_size) - changing either invalidates
        old storage automatically."""
        key = "|".join(self.diseases) + f"|{self.img_size}"
        return hashlib.md5(key.encode()).hexdigest()[:8]

    def path(self, name: str) -> Path:
        return self.processed_dir / f"{name}_{self.cache_tag}"

In [3]:
RANDOM_SEED = 42
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
DATA_DIR = Path("/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection")
DISEASES = ("Aortic enlargement", "Cardiomegaly", "Pleural thickening", "Pulmonary fibrosis")

data_config = DataConfig(
    raw_dir=DATA_DIR / "train",
    processed_dir=Path("processed"),
    csv_path=DATA_DIR / "train.csv",
    img_size=224,
    diseases=DISEASES,
)


In [4]:
# --- Raw annotations ---
def load_annotations(csv_path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} annotation rows across {df.image_id.nunique()} images")
    print(f"Number of nans in column:")
    print(df.isna().sum())
    return df


In [5]:
def load_dicom_as_array(dicom_path) -> np.ndarray:
    dicom = pydicom.dcmread(dicom_path)
    array = apply_voi_lut(dicom.pixel_array, dicom)
    if getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1":
        array = np.amax(array) - array
    array = array.astype(np.float32)
    array = cv2.normalize(array, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
    return array

In [6]:
# --- Per-image processing 
def load_and_resize_image(raw_path: Path, img_size: int) -> tuple[np.ndarray, int, int]:
    array = load_dicom_as_array(raw_path)
    orig_h, orig_w = array.shape
    resized = cv2.resize(array, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return resized.astype(np.uint8, copy=False), orig_h, orig_w

In [7]:
def build_disease_masks(annotations: pd.DataFrame, config: DataConfig,
                         orig_h: int, orig_w: int) -> dict[str, np.ndarray]:
    img_size = config.img_size
    scale_x, scale_y = img_size / orig_w, img_size / orig_h
    masks = {d: np.zeros((img_size, img_size), dtype=np.bool_) for d in config.diseases}

    for cls, x_min, y_min, x_max, y_max in annotations.itertuples(index=False):
        if cls not in config.disease_to_idx:
            continue
        x1 = max(int(x_min * scale_x), 0)
        x2 = min(int(x_max * scale_x), img_size)
        y1 = max(int(y_min * scale_y), 0)
        y2 = min(int(y_max * scale_y), img_size)
        if x2 <= x1 or y2 <= y1:
            continue
        masks[cls][y1:y2, x1:x2] = True

    return masks

In [8]:
def build_image_universe(df: pd.DataFrame) -> list[str]:
    return sorted(df.image_id.unique().tolist())

In [9]:
def get_or_build_storage(df: pd.DataFrame, config: DataConfig, max_workers: int = 8):
    """
    Returns (image_id_to_row, image_store, mask_store).
    If a cache matching this exact config (diseases + img_size, via cache_tag)
    already exists on disk, reopens it instead of reprocessing.
    """
    lookup_path = config.path("lookup").with_suffix(".json")
    images_path = config.path("images").with_suffix(".dat")
    masks_path = config.path("masks").with_suffix(".dat")

    if lookup_path.exists() and images_path.exists() and masks_path.exists():
        print(f"Reusing existing cache (tag={config.cache_tag})")
        with open(lookup_path) as f:
            image_id_to_row = json.load(f)
        n = len(image_id_to_row)
        s = config.img_size
        image_store = np.memmap(images_path, dtype=np.uint8, mode="r+", shape=(n, s, s))
        mask_store = np.memmap(masks_path, dtype=np.bool_, mode="r+",
                                shape=(n, config.num_classes, s, s))
        return image_id_to_row, image_store, mask_store

    print(f"Building new cache (tag={config.cache_tag})")
    config.processed_dir.mkdir(parents=True, exist_ok=True)

    universe = build_image_universe(df)
    image_id_to_row = {img_id: i for i, img_id in enumerate(universe)}
    n, s = len(universe), config.img_size

    image_store = np.memmap(images_path, dtype=np.uint8, mode="w+", shape=(n, s, s))
    mask_store = np.memmap(masks_path, dtype=np.bool_, mode="w+",
                            shape=(n, config.num_classes, s, s))

    grouped = dict(list(df.groupby("image_id")[["class_name", "x_min", "y_min", "x_max", "y_max"]]))

    def decode_one(image_id: str):
        raw_path = config.raw_dir / f"{image_id}.dicom"
        image, orig_h, orig_w = load_and_resize_image(raw_path, config.img_size)
        masks = build_disease_masks(grouped[image_id], config, orig_h, orig_w)
        return image_id, image, masks

    # Parallelize the expensive part (DICOM decode); write sequentially on
    # the main thread to avoid concurrent writes into the same memmap file.
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for image_id, image, masks in tqdm(
            executor.map(decode_one, universe), total=len(universe), desc="Preprocessing"
        ):
            row = image_id_to_row[image_id]
            image_store[row] = image
            for disease, mask in masks.items():
                mask_store[row, config.disease_to_idx[disease]] = mask

    image_store.flush()
    mask_store.flush()
    with open(lookup_path, "w") as f:
        json.dump(image_id_to_row, f)

    return image_id_to_row, image_store, mask_store

In [10]:
# build preprocessed storage 
df = load_annotations(data_config.csv_path)
image_id_to_row, image_store, mask_store = get_or_build_storage(df, data_config)

Loaded 67914 annotation rows across 15000 images
Number of nans in column:
image_id          0
class_name        0
class_id          0
rad_id            0
x_min         31818
y_min         31818
x_max         31818
y_max         31818
dtype: int64
Building new cache (tag=e9f34c79)


Preprocessing:   0%|          | 0/15000 [00:00<?, ?it/s]